# Stephens Lake — Mineral Saturation Index Analysis

This notebook calculates saturation indices (SI) for five minerals—**Calcite**, **Dolomite**, **Goethite**, **Pyrolusite**, and **Hydroxyapatite**—for every water sample in the Stephens Lake diel dataset using the PHREEQC geochemical model (via **PhreeqPy / IPhreeqc**) and the `phreeqc.dat` thermodynamic database.

**Interpretation of SI:**
- SI > 0 → water is supersaturated; the mineral tends to precipitate
- SI = 0 → mineral–solution equilibrium
- SI < 0 → water is undersaturated; the mineral tends to dissolve

## Outputs
| File | Description |
|------|-------------|
| `Stephens_Lake_{Season}_diel_SI.csv` | Full diel time series (all parameters + SI) per season |
| `Stephens_Lake_SI_panel.png` | 2×2 season panel plot (PNG, 150 dpi) |
| `Stephens_Lake_SI_panel.svg` | 2×2 season panel plot (SVG, vector) |

## Requirements
```
pip install phreeqpy pandas matplotlib
```
`phreeqc.dat` must be in the same directory as this notebook.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import phreeqpy.iphreeqc.phreeqc_dll as _phreeqc_dll
except ImportError as exc:
    raise ImportError(
        "phreeqpy is required. Install with:\n"
        "  pip install phreeqpy\n"
        "The IPhreeqc shared library is bundled with phreeqpy on most platforms."
    ) from exc

print("All imports OK.")

## Configuration

All file paths, mineral names, seasons, and plot styles are defined here.  
Adjust `DB_PATH` or `CSV_PATH` if the files are in a different location.

In [ ]:
# Paths – resolved relative to the notebook working directory
NOTEBOOK_DIR = Path().resolve()
DB_PATH  = NOTEBOOK_DIR / "phreeqc.dat"
CSV_PATH = 'output\\StephensLake_AllSeasons_Chemistry.csv'
OUT_DIR  = 'output'

assert DB_PATH.exists(),  f"Database not found: {DB_PATH}"
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

MINERALS = ["Calcite", "Dolomite", "Goethite", "Pyrolusite", "Hydroxyapatite"]
SEASONS  = ["Fall", "Winter", "Spring", "Summer"]

# Line style per mineral
MINERAL_STYLE: dict[str, dict] = {
    "Calcite":        {"color": "#1f77b4", "ls": "-",            "lw": 1.8},
    "Dolomite":       {"color": "#ff7f0e", "ls": "--",           "lw": 1.8},
    "Goethite":       {"color": "#8B0000", "ls": "-.",           "lw": 1.8},
    "Pyrolusite":     {"color": "#9467bd", "ls": ":",            "lw": 2.2},
    "Hydroxyapatite": {"color": "#2ca02c", "ls": (0,(3,1,1,1)), "lw": 1.8},
}

# Molecular weights (g/mol) for unit conversions
_MW = {
    "Si":    28.086,
    "SiO2":  60.084,
    "NO3":   62.004,
    "N":     14.007,
    "C":     12.011,
    "HCO3":  61.016,
}

# Raw CSV columns required to compute each mineral's SI.
# If any listed column is NaN for a sample, that mineral's SI is set to NaN
# so it is not plotted for that sample.
MINERAL_REQUIRED_COLS: dict[str, list[str]] = {
    "Calcite":        ["Calcium_mg_L-1",   "DIC_mg_L-1"],
    "Dolomite":       ["Calcium_mg_L-1",   "Magnesium_mg_L-1", "DIC_mg_L-1"],
    "Goethite":       ["Iron_ug_L-1",      "DO_mg_L-1"],
    "Pyrolusite":     ["Manganese_ug_L-1", "DO_mg_L-1"],
    "Hydroxyapatite": ["Calcium_mg_L-1",   "Phosphorus_mg_L-1"],
}

print("Configuration ready.")
print(f"  Database : {DB_PATH}")
print(f"  CSV      : {CSV_PATH}")

## 1. Data Loading

The CSV contains one row per 4-hour sampling interval across four diel campaigns (Fall, Winter, Spring, Summer). Key measured variables used in the PHREEQC calculations:

| Variable | Column | Units |
|----------|--------|-------|
| Temperature | `TEMP_degC` | °C |
| pH | `PH` | — |
| Dissolved O₂ | `DO_mg_L-1` | mg/L as O₂ |
| DIC | `DIC_mg_L-1` | mg/L as C |
| Major cations/anions | `Calcium_mg_L-1`, etc. | mg/L as element |
| Sulfate | `Sulfate_mg_L-1` | mg/L as SO₄ |
| Nitrate | `Nitrate_mg_L-1` | mg/L as NO₃ |
| Phosphorus | `Phosphorus_mg_L-1` | mg/L as P |
| Silicon | `Silicon_mg_L-1` | mg/L as Si |
| Iron | `Iron_ug_L-1` | µg/L as Fe |
| Manganese | `Manganese_ug_L-1` | µg/L as Mn |

Negative concentration values (below-detection-limit artefacts) are clipped to zero.

In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """Read CSV, parse datetimes, clip negative concentrations to zero."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    df["DATETIME"] = pd.to_datetime(df["DATETIME"])

    conc_cols = [c for c in df.columns
                 if any(tag in c for tag in ("_mg_L", "_ug_L"))]
    for col in conc_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").clip(lower=0)

    return df


df = load_data(CSV_PATH)
print(f"{len(df)} samples loaded")
print("Sample counts by season:")
print(df["Season"].value_counts().to_string())
df.head()

## 2. PHREEQC Input Template (Inlined)

The `build_phreeqc_input()` function constructs the complete PHREEQC input string for each water sample. Everything—SOLUTION block, SELECTED_OUTPUT block, and END keyword—is inlined in this single Python function.

### Unit-conversion rationale

PHREEQC converts mg/L concentrations to mol/kgw using the **gram formula weight** (`gfw_formula`) defined in `SOLUTION_MASTER_SPECIES` of `phreeqc.dat`. Several data columns require conversion:

| Element | phreeqc.dat gfw_formula | gfw (g/mol) | Data units | Conversion needed |
|---------|------------------------|-------------|------------|-------------------|
| Ca, Mg, Na, K, Fe, Mn, P | element | atomic mass | mg/L as element | **none** |
| S(6) | SO₄ | 96.06 | mg/L as SO₄ | **none** |
| N(5) | N | 14.007 | mg/L as NO₃ | × 14.007/62.004 |
| Si | SiO₂ | 60.084 | mg/L as Si | × 60.084/28.086 |
| C(4) | HCO₃ | 61.016 | mg/L as C | × 61.016/12.011 → mg/L as HCO₃ |
| O(0) | O | 15.999 | mg/L as O₂ | **none** (see note) |

> **O(0) note:** PHREEQC divides input by MW(O) = 15.999. Inputting 8 mg/L gives 5.0×10⁻⁴ mol O(0)/kgw = 2.5×10⁻⁴ mol O₂/kgw = **8 mg/L O₂** ✓

### Redox, iron, manganese, and phosphorus
Dissolved oxygen sets the redox state (pe). PHREEQC distributes total Fe between Fe²⁺ and Fe³⁺ based on pe, enabling Goethite SI (requires Fe³⁺) and Pyrolusite SI (two-electron MnO₂ reduction). Hydroxyapatite SI (Ca₅(PO₄)₃OH) depends on Ca, PO₄, and pH.

In [ ]:
def _safe(val, default: float = 0.0, minimum: float = 0.0) -> float:
    """Return a finite, clamped float safe for a PHREEQC input string."""
    try:
        v = float(val)
        return default if (np.isnan(v) or np.isinf(v)) else max(v, minimum)
    except (TypeError, ValueError):
        return default


def build_phreeqc_input(row: pd.Series, sol_num: int = 1) -> str:
    """
    Return the complete PHREEQC input string for one water sample.
    SOLUTION, SELECTED_OUTPUT, and END are all assembled in this function.
    """
    temp  = _safe(row.get("TEMP_degC"),              default=25.0)
    pH    = _safe(row.get("PH"),                     default=7.0)
    Ca    = _safe(row.get("Calcium_mg_L-1"))
    Mg    = _safe(row.get("Magnesium_mg_L-1"))
    Na    = _safe(row.get("Sodium_mg_L-1"))
    K     = _safe(row.get("Potassium_mg_L-1"))
    Cl    = _safe(row.get("Chloride_mg_L-1"))
    SO4   = _safe(row.get("Sulfate_mg_L-1"))
    NO3   = _safe(row.get("Nitrate_mg_L-1"))
    P     = _safe(row.get("Phosphorus_mg_L-1"), minimum=1e-9)
    Si_el = _safe(row.get("Silicon_mg_L-1"))
    Fe_ug = _safe(row.get("Iron_ug_L-1"))
    Mn_ug = _safe(row.get("Manganese_ug_L-1"))
    DIC   = _safe(row.get("DIC_mg_L-1"),   minimum=0.01)
    DO    = _safe(row.get("DO_mg_L-1"), default=8.0, minimum=0.01)

    NO3_as_N    = NO3   * (_MW["N"]    / _MW["NO3"])
    Si_as_SiO2  = Si_el * (_MW["SiO2"] / _MW["Si"])
    Fe_mg       = max(Fe_ug / 1_000.0, 1e-9)
    Mn_mg       = max(Mn_ug / 1_000.0, 1e-9)
    DIC_as_HCO3 = DIC   * (_MW["HCO3"] / _MW["C"])

    # Inlined PHREEQC template: assemble lines then join with newlines.
    # This avoids any ambiguity between Python string escapes and PHREEQC
    # newline characters when the notebook JSON is serialised.
    lines = [
        f"SOLUTION {sol_num}",
        f"    temp    {temp:.4f}",
        f"    pH      {pH:.4f}",
        f"    units   mg/L",
        f"    density 1.0",
        f"    Ca      {Ca:.6f}",
        f"    Mg      {Mg:.6f}",
        f"    Na      {Na:.6f}",
        f"    K       {K:.6f}",
        f"    Cl      {Cl:.6f}",
        f"    S(6)    {SO4:.6f}",
        f"    N(5)    {NO3_as_N:.9f}",
        f"    P       {P:.10f}",
        f"    Si      {Si_as_SiO2:.6f}",
        f"    Fe      {Fe_mg:.9f}",
        f"    Mn      {Mn_mg:.9f}",
        f"    C(4)    {DIC_as_HCO3:.6f}",
        f"    O(0)    {DO:.6f}",
        "",
        "SELECTED_OUTPUT",
        "    -si     Calcite Dolomite Goethite Pyrolusite Hydroxyapatite",
        "",
        "END",
    ]
    return "\n".join(lines) + "\n"


# Preview the PHREEQC input for the first sample
print("--- PHREEQC input for sample 1 ---")
print(build_phreeqc_input(df.iloc[0], sol_num=1))

## 3. IPhreeqc Runner

The `run_iphreeqc()` function creates a fresh `IPhreeqc` instance for each call (thread-safe, no state leakage between samples), loads the thermodynamic database, executes the input string, and parses the `SELECTED_OUTPUT` array returned by PhreeqPy.

The `get_selected_output_array()` method returns a list of lists:
- Row 0 → column header names (e.g., `si_Calcite`, `si_Dolomite`, …)
- Row 1 → corresponding numeric values for this solution

SI values are looked up by header name, so column order does not matter.

In [ ]:
def run_iphreeqc(input_str: str, db_path: Path) -> dict[str, float]:
    """
    Execute IPhreeqc on *input_str* with *db_path* as the thermodynamic database.
    Returns {mineral_name: SI_value}. Returns NaN where SI cannot be retrieved.
    """
    ip = _phreeqc_dll.IPhreeqc()
    ip.load_database(str(db_path))
    rc = ip.run_string(input_str)
    if rc != 0:
        warnings.warn(f"IPhreeqc returned code {rc}: {ip.get_error_string()}")

    arr = ip.get_selected_output_array()
    if len(arr) < 2:
        return {m: np.nan for m in MINERALS}

    header = [str(h).strip() for h in arr[0]]
    data   = list(arr[1])

    result: dict[str, float] = {}
    for mineral in MINERALS:
        key = f"si_{mineral}"
        try:
            v = float(data[header.index(key)])
            result[mineral] = v if np.isfinite(v) else np.nan
        except (ValueError, IndexError, TypeError):
            result[mineral] = np.nan
    return result


# Quick test on the first sample
test_si = run_iphreeqc(build_phreeqc_input(df.iloc[0], sol_num=1), DB_PATH)
print("SI for first sample:")
for mineral, si in test_si.items():
    print(f"  SI({mineral:16s}) = {si:+.4f}")

## 4. Batch Saturation-Index Calculation

`calculate_si()` iterates over every row, builds a PHREEQC input string, runs IPhreeqc, and collects the results. The returned DataFrame is the original CSV extended with five new SI columns: `Calcite`, `Dolomite`, `Goethite`, `Pyrolusite`, `Hydroxyapatite`.

In [ ]:
def calculate_si(df: pd.DataFrame, db_path: Path) -> pd.DataFrame:
    """
    Run IPhreeqc for every row in *df* and append five SI columns.
    Returns the original DataFrame extended with the mineral SI columns.

    After each PHREEQC run, any mineral whose required species columns are NaN
    in the raw data has its SI overridden to NaN so it is excluded from plots.
    """
    records: list[dict] = []
    n = len(df)
    for i, (idx, row) in enumerate(df.iterrows(), start=1):
        if i % 5 == 0 or i == n:
            print(f"  {i}/{n}", end="\r", flush=True)
        inp = build_phreeqc_input(row, sol_num=i)
        try:
            si = run_iphreeqc(inp, db_path)
        except Exception as exc:
            warnings.warn(f"Row {idx} ({row.get('DATETIME', '?')}): {exc}")
            si = {m: np.nan for m in MINERALS}
        # Mask SI to NaN where a required species was not measured
        for mineral, req_cols in MINERAL_REQUIRED_COLS.items():
            if any(pd.isna(row.get(col)) for col in req_cols):
                si[mineral] = np.nan
        si["_idx"] = idx
        records.append(si)

    print()
    si_df = pd.DataFrame(records).set_index("_idx")
    return df.join(si_df)


print("Running PHREEQC for all samples ...")
df_si = calculate_si(df, DB_PATH)
print("Done.")
df_si[["Season", "DATETIME"] + MINERALS].head(10)

## 5. Summary Statistics

Mean and standard deviation of each SI by season.

In [ ]:
print("Mean SI by season:")
display(df_si.groupby("Season")[MINERALS].mean().round(3))

print("\nStandard deviation of SI by season:")
display(df_si.groupby("Season")[MINERALS].std().round(3))

In [ ]:
missing = df_si[MINERALS].isna().sum()
if missing.any():
    print("Missing SI values (NaN) per mineral:")
    print(missing[missing > 0].to_string())
else:
    print("All SI values computed successfully — no missing values.")

## 6. CSV Export — Diel Time Series for All Parameters

One CSV file is written per season, containing **all original measurement columns** plus the five computed SI columns. Files are written to the same directory as the notebook.

In [ ]:
def export_csv(df: pd.DataFrame, out_dir: Path) -> None:
    """Write a complete diel time-series CSV (all parameters + SI) per season."""
    for season in SEASONS:
        sub = df[df["Season"] == season]
        if sub.empty:
            continue
        fpath = out_dir / f"Stephens_Lake_{season}_diel_SI.csv"
        sub.to_csv(fpath, index=False)
        print(f"  Written: {fpath.name}  ({len(sub)} rows)")


print("Exporting CSVs:")
export_csv(df_si, OUT_DIR)

## 7. Panel Plot — Diel Time Series of SI by Season

The `plot_panel()` function creates a 2×2 figure with one subplot per season showing the diel SI time series for all five minerals. A horizontal black line at **SI = 0** marks the saturation boundary. A single shared legend appears below all panels.

**Reading the plot:**
- **Calcite / Dolomite:** controlled by Ca, Mg, DIC, pH, and temperature. SI rises with pH during daytime photosynthesis.
- **Goethite (FeOOH):** controlled by Fe³⁺ and pH. Oxygenated water tends to be supersaturated.
- **Pyrolusite (MnO₂):** controlled by Mn²⁺, pe, and pH. SI responds to diel DO variation.
- **Hydroxyapatite (Ca₅(PO₄)₃OH):** controlled by Ca, phosphate, and pH.

Both PNG (150 dpi) and SVG (vector) are saved.

In [ ]:
def plot_panel(df: pd.DataFrame, out_dir: Path) -> None:
    """
    2x2 panel: one subplot per season with diel SI for all five minerals.
    Single shared legend below the panels. Saved as PNG and SVG.

    All four subplots share the same y-axis range (global min/max across all
    seasons and minerals, padded by 0.5 SI units).  Samples whose required
    species are NaN produce a gap in the line rather than a plotted point.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Shared y-axis range across all seasons
    si_vals = df[MINERALS].to_numpy(dtype=float)
    finite  = si_vals[np.isfinite(si_vals)]
    if finite.size:
        pad   = 0.5
        y_min = finite.min() - pad
        y_max = finite.max() + pad
    else:
        y_min, y_max = -5.0, 5.0

    legend_handles = []
    for ax, season in zip(axes.flat, SEASONS):
        sub = df[df["Season"] == season].sort_values("DATETIME")
        if sub.empty:
            ax.set_title(f"{season} - no data", fontsize=12)
            continue

        for mineral in MINERALS:
            st = MINERAL_STYLE[mineral]
            line, = ax.plot(
                sub["DATETIME"], sub[mineral],
                color=st["color"], linestyle=st["ls"], linewidth=st["lw"],
                marker="o", markersize=4, label=mineral,
            )
            if season == SEASONS[0]:
                legend_handles.append(line)

        ax.axhline(0, color="k", lw=0.9, alpha=0.55)
        ax.set_ylim(y_min, y_max)
        ax.set_title(season, fontsize=13, fontweight="bold")
        ax.set_ylabel("Saturation Index", fontsize=10)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=7))
        ax.tick_params(axis="x", labelsize=8)
        ax.grid(True, alpha=0.3, lw=0.5)

    for ax in axes.flat:
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    fig.tight_layout(rect=[0, 0.06, 1, 1])
    fig.legend(
        handles=legend_handles,
        labels=MINERALS,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.0),
        ncol=len(MINERALS),
        fontsize=10,
        framealpha=0.9,
        edgecolor="0.7",
    )

    for ext in ("png", "svg"):
        fpath = out_dir / f"Stephens_Lake_SI_panel.{ext}"
        fig.savefig(fpath, dpi=150, bbox_inches="tight")
        print(f"  Saved: {fpath.name}")

    plt.show()


print("Generating panel plot ...")
plot_panel(df_si, OUT_DIR)

## Notes on Data Quality

1. **Nitrate:** Many Fall and Winter rows have missing Nitrate; treated as zero (negligible effect on SI).
2. **Summer DO / pH:** Inspect Summer data — DO and pH values appear identical for some rows, possibly indicating a sensor issue.
3. **Trace metals:** Fe and Mn near detection limit (< 1 µg/L) may produce very negative Goethite/Pyrolusite SI.
4. **Hydroxyapatite:** SI depends strongly on phosphate; missing or near-zero phosphate rows may show large negative SI.
5. **Charge balance:** IPhreeqc warnings flag solutions with large ion balance errors (> ±5 %) where SI values may be less reliable.